# Centroid tie-break + Semantic disambig

После того как `popularity` уступил `count`, пробуем два более информативных способа выбирать суффикс при коллизии.

**A. Centroid tie-break.** Внутри кластера коллизий считаем центроид латентных векторов энкодера и сортируем товары по близости к нему. Ближайший к центроиду — `cnt=0`. GPT2 чаще предсказывает `cnt=0`, и теперь это «прототип» кластера, а не случайный/популярный товар.

**B. Semantic disambig.** Вместо порядкового `cnt` квантизуем **residual** (то, что осталось от латента после вычитания всех L кодбуков) глобальным k-means на K кодов. Один общий кодбук на все коллизионные кластеры → `cnt=k` несёт переносимый смысл («такое-то отклонение от прототипа»), а не порядковый номер. GPT2 теперь может выучить условные распределения по 5-му токену.

Оба варианта используют те же чекпоинты RQ-VAE, что и предыдущий эксперимент. Сравнение идёт против `count` (выигравшего в твоём прошлом A/B).

In [ ]:
import os, glob, math, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from tqdm import tqdm

for p in (r'C:\Users\Admin\Documents\GitHub\claude_plum',
          '/kaggle/input/datasets/qwerte123'):
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)
import sid_utils
print('sid_utils loaded from', sid_utils.__file__)

In [ ]:
KAGGLE_BASE  = '/kaggle/input/datasets/qwerte123/hetero-data-updated-diffemb'
LOCAL_BASE   = r'C:\Users\Admin\Documents\GitHub\claude_plum\data'
BASE         = KAGGLE_BASE if os.path.exists(KAGGLE_BASE) else LOCAL_BASE

DATA_PATH    = os.path.join(BASE, 'heterodata_object12_updated.pt')
SEQ_PATH     = os.path.join(BASE, 'sequential_data.txt')
CKPT_DIR_IMP = os.path.join(BASE, 'checkpoints_improved')

SAVE_DIR = ('/kaggle/working/exp_centroid_semantic'
            if os.path.exists('/kaggle')
            else os.path.join(LOCAL_BASE, 'exp_centroid_semantic'))
os.makedirs(SAVE_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

df = torch.load(DATA_PATH, weights_only=False, map_location='cpu')
embeds = df['item'].x
n_items = embeds.shape[0]
print(f'items: {n_items}  embed_dim: {embeds.shape[1]}')

sequences  = sid_utils.load_sequences(SEQ_PATH)
popularity = sid_utils.compute_item_popularity(sequences, n_items)

## RQ-VAE classes (копия из предыдущих ноутбуков)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        layers, dims = [], [input_dim] + list(hidden_dims) + [output_dim]
        for i, (a, b) in enumerate(zip(dims[:-1], dims[1:])):
            layers.append(nn.Linear(a, b, bias=True))
            if i < len(dims) - 2:
                layers.append(nn.LayerNorm(b))
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class EMACodebook(nn.Module):
    def __init__(self, codebook_size, emb_dim, beta=0.25, ema_decay=0.99, epsilon=1e-4):
        super().__init__()
        self.codebook_size = codebook_size; self.beta = beta
        self.ema_decay = ema_decay; self.epsilon = epsilon
        emb = F.normalize(torch.randn(codebook_size, emb_dim), p=2, dim=1)
        self.register_buffer('emb', emb)
        self.register_buffer('ema_count', torch.ones(codebook_size))
        self.register_buffer('ema_weight', emb.clone())
        self.register_buffer('initialized', torch.zeros(1, dtype=torch.bool))
    def forward(self, x):
        x_n = F.normalize(x, p=2, dim=1)
        code_n = F.normalize(self.emb, p=2, dim=1)
        ids = (1.0 - x_n @ code_n.T).argmin(dim=1)
        emb = self.emb[ids]
        return self.beta * F.mse_loss(x, emb.detach()), x + (emb - x).detach(), ids

class _RQVAEBase(nn.Module):
    def __init__(self, inp_size, hidden_sizes, embed_dim, n_layers,
                 codebook_size=256, beta=0.25, gamma=0.1, ema_decay=0.99, **kwargs):
        super().__init__()
        self.n_layers = n_layers
        self.enc = Encoder(inp_size, hidden_sizes, embed_dim)
        self.dec = Encoder(embed_dim, hidden_sizes[::-1], inp_size)
        self.codebooks = nn.ModuleList([
            EMACodebook(codebook_size, embed_dim, beta=beta, ema_decay=ema_decay)
            for _ in range(n_layers)
        ])
    def forward(self, x):
        x_n = F.normalize(x, p=2, dim=1)
        r, sids = self.enc(x_n), []
        for cb in self.codebooks:
            _, emb_st, ids = cb(r)
            r = r - emb_st.detach()
            sids.append(ids)
        return {'sids': sids}

class RQVAE_Improved(_RQVAEBase):
    def __init__(self, *args, temperature=0.07, **kwargs):
        super().__init__(*args, **kwargs); self.temperature = temperature

import re
def infer_hidden_sizes(state_dict):
    items = [(k, v) for k, v in state_dict.items()
             if k.startswith('enc.net.') and k.endswith('.weight') and v.dim() == 2]
    items.sort(key=lambda kv: int(re.search(r'enc\.net\.(\d+)\.weight', kv[0]).group(1)))
    return [v.shape[0] for _, v in items][:-1]

def load_rqvae(ckpt_path, model_class, inp_size, device):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    hp = ckpt['hparams']
    hs = infer_hidden_sizes(ckpt['model_state'])
    kw = {k: hp[k] for k in ('embed_dim','n_layers','codebook_size','beta','gamma','ema_decay') if k in hp}
    if 'temperature' in hp: kw['temperature'] = hp['temperature']
    model = model_class(inp_size=inp_size, hidden_sizes=hs, **kw).to(device)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    return model, hp

best_imp_path = sorted(glob.glob(os.path.join(CKPT_DIR_IMP, 'rqvae_improved_s*.pt')))[-1]
rqvae, hp = load_rqvae(best_imp_path, RQVAE_Improved, embeds.shape[1], device)
print('Using', os.path.basename(best_imp_path), 'L=', hp['n_layers'], 'K=', hp['codebook_size'])

## Извлекаем base SID, латенты и residual

- `base[i]` — кортеж длины L = 4 (текущая конфигурация).
- `Z[i]` — выход энкодера, по нему считаем центроиды кластеров.
- `R[i]` — residual после вычитания всех L кодбуков; именно его кластеризуем для семантического disambig.


In [ ]:
base = sid_utils.encode_base_sids(rqvae, embeds, device)
Z, R = sid_utils.encode_latents_and_residuals(rqvae, embeds, device)
print('Z:', tuple(Z.shape), 'R:', tuple(R.shape))

coll_mask = sid_utils.build_collision_mask(base)
print(f'items in any collision cluster: {coll_mask.sum()} / {len(base)}')

In [ ]:
# Семантический disambig: глобальный k-means на residuals только для коллизионных товаров.
# K = max_dupe (или фиксируем; здесь авто = max_dupe для baseline)
from collections import Counter
max_dupe_obs = max(Counter(base).values())
K_SEMANTIC = max(8, max_dupe_obs)
print(f'max observed cluster size = {max_dupe_obs}  ->  K_SEMANTIC = {K_SEMANTIC}')

semantic_codes = sid_utils.kmeans_residual_codes(R, K_SEMANTIC, coll_mask, n_iter=25, seed=0)
print(f'semantic_codes: {len(semantic_codes)}  unique used codes: {len(set(semantic_codes))}')

## Сборка трёх вариантов SID

`count` (baseline) / `centroid` / `semantic` — все на одном RQ-VAE.

In [ ]:
PAD_ID, BOS_ID = 0, 1
MAX_HIST_LEN = 20
D_MODEL, N_HEADS, N_LAYERS_GPT, DROPOUT = 256, 8, 4, 0.1
BATCH_SIZE, LR, WARMUP_STEPS = 256, 1e-3, 500
N_EPOCHS = 30
BEAM_SIZE = 32
EVAL_KS = [1, 5, 10, 20]

def build_pipeline(tie_break):
    if tie_break == 'centroid':
        full, sid2item, max_dupe = sid_utils.assign_sids(base, tie_break='centroid', latents=Z)
    elif tie_break == 'semantic':
        full, sid2item, max_dupe = sid_utils.assign_sids(base, tie_break='semantic',
                                                          semantic_codes=semantic_codes)
    else:
        full, sid2item, max_dupe = sid_utils.assign_sids(base, tie_break='count')
    L, K = hp['n_layers'], hp['codebook_size']
    n_levels = L + 1
    lev_off = [2 + l * K for l in range(L)] + [2 + L * K]
    vocab = 2 + L * K + max_dupe
    def item_to_tokens(i):
        sid = full[i]
        return [sid[l] + lev_off[l] for l in range(n_levels)]
    def history_to_tokens(ids):
        toks = [BOS_ID]
        for iid in ids: toks.extend(item_to_tokens(int(iid)))
        return toks
    trie = sid_utils.build_trie(sid2item)
    return dict(full=full, sid2item=sid2item, max_dupe=max_dupe,
                n_levels=n_levels, lev_off=lev_off, vocab=vocab,
                item_to_tokens=item_to_tokens, history_to_tokens=history_to_tokens,
                trie=trie)

pipes = {tb: build_pipeline(tb) for tb in ('count', 'centroid', 'semantic')}
for tb, p in pipes.items():
    print(f'{tb:9s}  vocab={p["vocab"]:5d}  max_dupe={p["max_dupe"]:3d}  unique_sids={len(p["sid2item"])}')

In [ ]:
# Sanity: насколько cnt=0 совпадает между стратегиями?
def cnt0_set(pipe):
    return {i for i, sid in pipe['full'].items() if sid[-1] == 0}

S_count    = cnt0_set(pipes['count'])
S_centroid = cnt0_set(pipes['centroid'])
S_semantic = cnt0_set(pipes['semantic'])
print(f'|cnt=0 count|     = {len(S_count)}')
print(f'|cnt=0 centroid|  = {len(S_centroid)}   overlap with count: {len(S_count & S_centroid)}')
print(f'|cnt=0 semantic|  = {len(S_semantic)}   overlap with count: {len(S_count & S_semantic)}')

## GPT2Rec + train/eval (то же что в `popularity_tiebreak_and_spearman.ipynb`)

In [ ]:
class GPT2Rec(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_seq_len, n_levels, dropout=0.1):
        super().__init__()
        self.d_model, self.n_levels, self.max_seq_len = d_model, n_levels, max_seq_len
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.lvl_emb = nn.Embedding(n_levels, d_model)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=4*d_model,
                                         dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)
                if m.padding_idx is not None: m.weight.data[m.padding_idx].zero_()
    def _level_ids(self, T, dev):
        ids = torch.zeros(T, dtype=torch.long, device=dev)
        for pos in range(1, T): ids[pos] = (pos - 1) % self.n_levels
        return ids.unsqueeze(0)
    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        lvl = self._level_ids(T, x.device).expand(B, -1)
        h = self.tok_emb(x) + self.pos_emb(pos) + self.lvl_emb(lvl)
        causal = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        pad_m = (x == PAD_ID)
        for layer in self.transformer.layers:
            h = layer(h, src_mask=causal, src_key_padding_mask=pad_m)
            h = h.masked_fill(pad_m.unsqueeze(-1), 0.0)
        if self.transformer.norm is not None: h = self.transformer.norm(h)
        return self.lm_head(self.ln_f(h))

class RecDataset(Dataset):
    def __init__(self, samples, max_hist_len, n_levels, full_supervision, item_to_tokens, history_to_tokens):
        self.samples=samples; self.max_hist_len=max_hist_len; self.n_levels=n_levels
        self.full_supervision=full_supervision
        self.item_to_tokens=item_to_tokens; self.history_to_tokens=history_to_tokens
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        ctx, tgt = self.samples[idx]
        ctx = ctx[-self.max_hist_len:]
        inp = self.history_to_tokens(ctx) + self.item_to_tokens(tgt)
        inp = torch.tensor(inp, dtype=torch.long)
        lbl = inp[1:].clone()
        if not self.full_supervision: lbl[:-self.n_levels] = -100
        lbl = torch.cat([lbl, torch.tensor([-100])])
        return inp, lbl

def collate_fn(batch):
    inps, lbls = zip(*batch)
    M = max(x.shape[0] for x in inps)
    pi, pl = [], []
    for inp, lbl in zip(inps, lbls):
        pad = M - inp.shape[0]
        pi.append(F.pad(inp, (pad, 0), value=PAD_ID))
        pl.append(F.pad(lbl, (pad, 0), value=-100))
    return torch.stack(pi), torch.stack(pl)

@torch.no_grad()
def beam_search(model, ctx_tok, trie, beam_size, device, level_offsets):
    model.eval()
    n_levels = len(level_offsets); ctx = ctx_tok.to(device)
    logits0 = model(ctx.unsqueeze(0))[0, -1, :]
    beams = [(logits0[c + level_offsets[0]].item(), (c,), sub) for c, sub in trie.items()]
    beams.sort(key=lambda x: -x[0]); beams = beams[:beam_size]
    for lvl in range(1, n_levels):
        code_toks = torch.tensor(
            [[codes[l] + level_offsets[l] for l in range(len(codes))] for _, codes, _ in beams],
            device=device)
        batch = torch.cat([ctx.unsqueeze(0).expand(len(beams), -1), code_toks], dim=1)
        logits = model(batch)[:, -1, :]
        new_beams, is_last = [], (lvl == n_levels - 1)
        for i, (score, codes, node) in enumerate(beams):
            for c, child in node.items():
                ns = score + logits[i, c + level_offsets[lvl]].item()
                new_beams.append((ns, child) if is_last else (ns, codes + (c,), child))
        new_beams.sort(key=lambda x: -x[0])
        if is_last: return new_beams
        beams = new_beams[:beam_size]
    return []

def evaluate(samples, model, trie, level_offsets, history_to_tokens, beam_size, ks, device, max_hist_len):
    hits, ndcg, total = defaultdict(int), defaultdict(float), 0
    for ctx, tgt in tqdm(samples, leave=False):
        ctx_tok = torch.tensor(history_to_tokens(ctx[-max_hist_len:]), dtype=torch.long)
        ranked = beam_search(model, ctx_tok, trie, beam_size, device, level_offsets)
        ranked_ids = [iid for _, iid in ranked]
        for k in ks:
            top_k = ranked_ids[:k]
            if tgt in top_k:
                hits[k] += 1; ndcg[k] += 1.0 / math.log2(top_k.index(tgt) + 2)
        total += 1
    return {**{f'Recall@{k}': round(hits[k]/total, 4) for k in ks},
            **{f'NDCG@{k}':   round(ndcg[k]/total, 4) for k in ks}}

In [ ]:
hist = df['user', 'rated', 'item'].history
def make_split(split_key, padded=False):
    item_ids = hist[split_key]['item_ID']; item_next = hist[split_key]['item_ID_next']
    out = []
    for u in range(len(item_next)):
        ctx = [int(x) for x in (item_ids[u].tolist() if padded else item_ids[u]) if int(x) >= 0]
        tgt = int(item_next[u])
        if not ctx or tgt < 0 or tgt >= n_items: continue
        out.append((ctx, tgt))
    return out

samples_train = make_split('train', padded=False)
samples_val   = make_split('valid', padded=True)
samples_test  = make_split('test',  padded=True)
print(f'train={len(samples_train)}  val={len(samples_val)}  test={len(samples_test)}')

In [ ]:
def train_and_eval(pipe, n_epochs=N_EPOCHS, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    max_seq = 1 + (MAX_HIST_LEN + 1) * pipe['n_levels']
    ds_tr = RecDataset(samples_train, MAX_HIST_LEN, pipe['n_levels'], True,
                       pipe['item_to_tokens'], pipe['history_to_tokens'])
    ds_va = RecDataset(samples_val,   MAX_HIST_LEN, pipe['n_levels'], False,
                       pipe['item_to_tokens'], pipe['history_to_tokens'])
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
    dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    model = GPT2Rec(pipe['vocab'], D_MODEL, N_HEADS, N_LAYERS_GPT, max_seq, pipe['n_levels'], DROPOUT).to(device)
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = n_epochs * len(dl_tr)
    sch = optim.lr_scheduler.LambdaLR(opt, lambda s: (
        s / max(1, WARMUP_STEPS) if s < WARMUP_STEPS
        else max(0.05, 0.5 * (1.0 + math.cos(math.pi * (s - WARMUP_STEPS) / max(1, total - WARMUP_STEPS))))))

    best_val = float('inf'); best_state = None
    for epoch in range(1, n_epochs + 1):
        model.train(); tr_loss, n = 0.0, 0
        for inp, lbl in dl_tr:
            inp, lbl = inp.to(device), lbl.to(device)
            loss = F.cross_entropy(model(inp).view(-1, pipe['vocab']), lbl.view(-1), ignore_index=-100)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step()
            tr_loss += loss.item(); n += 1
        model.eval(); va, m = 0.0, 0
        with torch.no_grad():
            for inp, lbl in dl_va:
                inp, lbl = inp.to(device), lbl.to(device)
                va += F.cross_entropy(model(inp).view(-1, pipe['vocab']), lbl.view(-1), ignore_index=-100).item(); m += 1
        va /= max(1, m)
        if va < best_val: best_val = va; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if epoch == 1 or epoch % 5 == 0 or epoch == n_epochs:
            print(f'  ep {epoch:3d}  train_loss={tr_loss/max(1,n):.4f}  val_loss={va:.4f}  best={best_val:.4f}')

    model.load_state_dict(best_state)
    res = evaluate(samples_test, model, pipe['trie'], pipe['lev_off'],
                   pipe['history_to_tokens'], BEAM_SIZE, EVAL_KS, device, MAX_HIST_LEN)
    res['val_loss_best'] = round(best_val, 4)
    return res

## A/B/C: count vs centroid vs semantic

Один и тот же RQ-VAE, одинаковый seed, одинаковая GPT2Rec. Меняется только `tie_break`.

In [ ]:
results = {}
for tb in ('count', 'centroid', 'semantic'):
    print(f'\n=== tie_break = {tb} ===')
    results[tb] = train_and_eval(pipes[tb], seed=0)
    print(results[tb])

abc_df = pd.DataFrame(results).T
abc_df.to_csv(os.path.join(SAVE_DIR, 'tiebreak_abc.csv'))
print('\n=== Comparison ===')
print(abc_df)
for tb in ('centroid', 'semantic'):
    delta = (abc_df.loc[tb] - abc_df.loc['count']).round(4)
    print(f'\nDelta ({tb} - count):')
    print(delta)

## Что смотреть

**Если `centroid` обгоняет `count`:** наша гипотеза «cnt=0 должен быть прототипом, а не случайным товаром» подтверждается. Эффект ожидаем сильнее на больших K (Recall@10/20), потому что центроид — *семантически усреднённый* выбор и не смещает в популярные.

**Если `semantic` обгоняет всех:** глобальный residual-кодбук передаёт переносимый сигнал между кластерами, и GPT2 учится *условному* распределению на 5-м токене. Это самое интересное: 5-й код перестаёт быть «мусорным» и становится полноценным уровнем.

**Если `semantic` хуже `count`:** возможные причины:
1. K_SEMANTIC слишком велик — GPT2 не успевает выучить редкие коды. Уменьшить до 8.
2. Residual слишком шумный после 4 уровней. Стоит пробовать кластеризовать `Z` (не residual), что приближает идею к centroid-tie-break, но с явным кодбуком.
3. Несовместимость `semantic_codes[i]` и текущего GPT2 архитектурно — разные товары в одном кластере получают непоследовательные коды, и beam search через trie не покрывает все ID. (Реализация выше делает локальную дедупликацию `+1` — проверь, что она не раздувает `max_dupe` сильно).

**Sanity-проверка для отчёта:**
- Сравни `|cnt=0 count|` vs `|cnt=0 centroid|` vs `|cnt=0 semantic|` в cell-10. Между centroid и count overlap должен быть низкий (~50–70%), иначе разница в стратегии не достигла данных.
- Если `train_and_eval` под semantic выдаёт больший `vocab` — это ожидаемо: K_SEMANTIC > max_dupe возможен. Это не баг, но это *увеличивает* размер модели и не совсем честный A/B. Чтобы быть совсем чистым, форсировать K_SEMANTIC = max_dupe_obs.